In [ ]:
# Imports for data handling, visualization, and modeling
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

***1. LOAD DATASET***

**Master folder**

In [ ]:
# Load master data tables
products = pd.read_csv("Master/products.csv")
geography = pd.read_csv("Master/geography.csv")
customers = pd.read_csv("Master/customers.csv")
promotions = pd.read_csv("Master/promotions.csv")

**Transaction**

In [ ]:
# Load transaction tables
order_item = pd.read_csv("Transaction/order_items.csv", dtype={'promo_id_2': str})
orders = pd.read_csv("Transaction/orders.csv")
payments = pd.read_csv("Transaction/payments.csv")
returns = pd.read_csv("Transaction/returns.csv")
reviews = pd.read_csv("Transaction/reviews.csv")
shipments = pd.read_csv("Transaction/shipments.csv")

**Analytical**

In [ ]:
# Load analytical sales table
sales = pd.read_csv("sales.csv")

**Operational**

In [ ]:
# Load operational tables
inventory = pd.read_csv("Operational/inventory.csv")
web_traffic = pd.read_csv("Operational/web_traffic.csv")

***2. Preprocessing Data***

In [ ]:
def preprocessing(sales_df, traffic_df, inv_df, promotions_df):
    """
    Hàm tiền xử lý dữ liệu.
    """
    df = sales_df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    # --- 1. CYCLICAL ENCODING ---
    # Tháng (chu kỳ 12)
    df['month_sin'] = np.sin(2 * np.pi * df['Date'].dt.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['Date'].dt.month / 12)
    
    # Thứ (chu kỳ 7)
    df['dow_sin'] = np.sin(2 * np.pi * df['Date'].dt.dayofweek / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['Date'].dt.dayofweek / 7)
    
    # Ngày trong tháng (chu kỳ 31)
    df['dom_sin'] = np.sin(2 * np.pi * df['Date'].dt.day / 31)
    df['dom_cos'] = np.cos(2 * np.pi * df['Date'].dt.day / 31)

    # --- 2. EVENT FEATURES ---
    # Vn Holidays
    holidays = pd.to_datetime(['2022-01-01', '2022-04-30', '2022-05-01', '2022-09-02'])
    df['days_to_holiday'] = df['Date'].apply(
        lambda x: min([(h - x).days for h in holidays if (h - x).days >= 0] or [365])
    )
    
    # --- 3. WEB TRAFFIC (lag 1 năm) ---
    traffic = traffic_df.copy()
    traffic['date'] = pd.to_datetime(traffic['date'])
    daily_traffic = traffic.groupby('date')['sessions'].sum().reset_index()
    daily_traffic['sessions_lag_365'] = daily_traffic['sessions'].shift(365)
    
    df = df.merge(daily_traffic[['date', 'sessions_lag_365']], 
                  left_on='Date', right_on='date', how='left').drop(columns=['date'])

    # --- 4. INVENTORY (lag 1 tháng) ---
    inv = inv_df.copy()
    inv['snapshot_date'] = pd.to_datetime(inv['snapshot_date'])
    monthly_inv = inv.groupby([inv['snapshot_date'].dt.year, inv['snapshot_date'].dt.month]).agg({
        'fill_rate': 'mean',
        'stockout_days': 'mean'
    }).rename_axis(['year', 'month']).reset_index()
    
    # Ảnh hưởng của tồn kho thág trước
    monthly_inv['fill_rate_prev_month'] = monthly_inv['fill_rate'].shift(1)
    
    df['tmp_year'], df['tmp_month'] = df['Date'].dt.year, df['Date'].dt.month
    df = df.merge(monthly_inv[['year', 'month', 'fill_rate_prev_month']], 
                  left_on=['tmp_year', 'tmp_month'], right_on=['year', 'month'], how='left')
    
    # --- 5. LOG VALUE ---
    if 'Revenue' in df.columns:
        df['target'] = np.log1p(df['Revenue'])
    
    # Xóa cột phụ
    drop_cols = ['tmp_year', 'tmp_month', 'year', 'month']
    return df.drop(columns=[c for c in drop_cols if c in df.columns]).fillna(0)

Data = preprocessing(sales, web_traffic, inventory, promotions)

In [ ]:
def add_lunar_features(df):
    # Thời gian mùng 1 Tết âm lịch từ 2012 đến 2024
    tet_dates = pd.to_datetime([
        '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19', 
        '2016-02-08', '2017-01-28', '2018-02-16', '2019-02-05', 
        '2020-01-25', '2021-02-12', '2022-02-01', '2023-01-22', '2024-02-10'
    ])
    
    # Số ngày còn lại đến Tết gần nhất
    def get_days_to_tet(current_date):
        future_tets = tet_dates[tet_dates >= current_date]
        if len(future_tets) > 0:
            return (future_tets[0] - current_date).days
        return 365 # Giá trị mặc định nếu vượt ngưỡng

    df['days_to_tet'] = df['Date'].apply(get_days_to_tet)
    
    # Binary values cho mùa Tết (21 ngày trước Tết)
    df['is_tet_season'] = df['days_to_tet'].apply(lambda x: 1 if 0 < x <= 21 else 0)
    
    # Cyclical Encoding cho ngày đến Tết (Chu kỳ 365 ngày)
    df['tet_proximity_sin'] = np.sin(2 * np.pi * df['days_to_tet'] / 365)
    df['tet_proximity_cos'] = np.cos(2 * np.pi * df['days_to_tet'] / 365)
    
    return df

Data = add_lunar_features(Data)

In [ ]:
#Hàm chuyển về đơn vị tiền tệ thực tế
def get_final_predictions(log_preds):
    """
    Chuyển đổi dự báo từ miền Log về miền tiền tệ thực tế.
    """
    # np.expm1 tương đương với exp(x) - 1
    final_preds = np.expm1(log_preds)
    return np.maximum(0, final_preds)

In [ ]:
def process_web_traffic(sales_df, traffic_df):
    traffic = traffic_df.copy()
    traffic['date'] = pd.to_datetime(traffic['date'])
    
    # Gom nhóm theo ngày để có tổng sessions và trung bình bounce_rate mỗi ngày
    daily_traffic = traffic.groupby('date').agg({
        'sessions': 'sum',
        'bounce_rate': 'mean'
    }).reset_index()

    # Lag 365
    daily_traffic['sessions_lag_365'] = daily_traffic['sessions'].shift(365)
    daily_traffic['bounce_rate_lag_365'] = daily_traffic['bounce_rate'].shift(365)
    
    # Tính EMA (7 ngày) để nắm xu hướng trend gần nhất của năm ngoái
    daily_traffic['sessions_trend_lag'] = daily_traffic['sessions_lag_365'].ewm(span=7).mean()


    sales_df = sales_df.merge(
        daily_traffic[['date', 'sessions_lag_365', 'bounce_rate_lag_365', 'sessions_trend_lag']], 
        left_on='Date', right_on='date', how='left'
    )
    
    return sales_df.drop(columns=['date']).ffill().fillna(0)

def process_inventory(sales_df, inv_df):
    inv = inv_df.copy()
    inv['snapshot_date'] = pd.to_datetime(inv['snapshot_date'])
    
    # Gom nhóm theo tháng để có fill_rate và stockout_days trung bình mỗi tháng, đồng thời tính tổng reorder_flag mỗi tháng
    monthly_inv = inv.groupby([inv['snapshot_date'].dt.year, inv['snapshot_date'].dt.month]).agg({
        'fill_rate': 'mean',
        'stockout_days': 'mean',
        'reorder_flag': 'sum' # Tổng số sản phẩm cần nhập thêm
    }).rename_axis(['year', 'month']).reset_index()

    #  Lag 1 tháng để biết ảnh hưởng của tồn kho tháng trước.
    monthly_inv['fill_rate_lag_1m'] = monthly_inv['fill_rate'].shift(1)
    monthly_inv['stockout_intensity'] = monthly_inv['stockout_days'].shift(1)

    # Tạo key để merge
    sales_df['tmp_year'] = sales_df['Date'].dt.year
    sales_df['tmp_month'] = sales_df['Date'].dt.month
    
    sales_df = sales_df.merge(
        monthly_inv[['year', 'month', 'fill_rate_lag_1m', 'stockout_intensity']], 
        left_on=['tmp_year', 'tmp_month'], right_on=['year', 'month'], how='left'
    )
    
    return sales_df.drop(columns=['tmp_year', 'tmp_month', 'year', 'month']).ffill().fillna(0)

Data = process_web_traffic(Data, web_traffic)
Data = process_inventory(Data, inventory)

In [ ]:
# --- TẠO KHUNG THỜI GIAN TƯƠNG LAI ---
future_dates = pd.date_range(start='2023-01-01', end='2024-07-01', freq='D')
future_df = pd.DataFrame({'Date': future_dates})

# --- ĐIỀN FEATURE CHO FUTURE_DF ---
future_df = add_lunar_features(future_df)
future_df = preprocessing(future_df, web_traffic, inventory, promotions)
future_df = process_web_traffic(future_df, web_traffic)
future_df = process_inventory(future_df, inventory)

# Chỉnh sửa lại tên cột sessions nếu bị trùng sau khi merge nhiều lần
if 'sessions_lag_365_x' in future_df.columns:
    future_df = future_df.rename(columns={'sessions_lag_365_x': 'sessions_lag_365'})

# Đảm bảo định dạng datetime cho bảng promotions
promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])

# Tạo một tập hợp (set) tất cả các ngày có khuyến mãi trong lịch sử
all_promo_dates = set()
for s, e in promotions[['start_date', 'end_date']].itertuples(index=False):
    all_promo_dates.update(pd.date_range(s, e))

# Thêm cột is_promotion vào bảng Data (Train)
Data['is_promotion'] = Data['Date'].isin(all_promo_dates).astype(int)

***3. Prediction Promotion Probability***

In [ ]:
# --- ĐỊNH NGHĨA LẠI FEATURES ---
# Xây dựng trên chính bộ features cyclical
promo_features_cyclical = [
    'month_sin', 'month_cos', 
    'dow_sin', 'dow_cos', 
    'dom_sin', 'dom_cos', 
    'days_to_tet', 'is_tet_season'
]

# --- CHUẨN BỊ DỮ LIỆU ---
# Train trên dữ liệu năm cũ
train_mask = (Data['Date'] <= '2022-12-31')
X_train_promo = Data.loc[train_mask, promo_features_cyclical]
y_train_promo = Data.loc[train_mask, 'is_promotion'].fillna(0)

# --- SCALE VÀ FIT ---
scaler_promo = StandardScaler()
X_train_scaled = scaler_promo.fit_transform(X_train_promo)

log_model = LogisticRegression(
    C=0.01,
    max_iter=1000,
    class_weight='balanced',
    penalty='l2',
    random_state=42
)
log_model.fit(X_train_scaled, y_train_promo)
print("Huấn luyện xong mô hình Promo Probability với Features Cyclical.")

# --- DỰ BÁO CHO TƯƠNG LAI ---
# Đảm bảo future_df đã có đủ các cột trong promo_features_cyclical
X_future_promo = future_df[promo_features_cyclical].fillna(0)
X_future_scaled = scaler_promo.transform(X_future_promo)

# Dự báo xác suất
future_df['promo_probability'] = log_model.predict_proba(X_future_scaled)[:, 1]
Data['promo_probability'] = log_model.predict_proba(scaler_promo.transform(Data[promo_features_cyclical].fillna(0)))[:, 1]
print("Đã xây dựng xong promo_probability cho tập Test.")

***4. Train Model***

In [ ]:
# Tách revenue và cogs
Data['target_rev'] = Data['target']  
Data['target_cogs'] = np.log1p(Data['COGS'].clip(lower=0))

# Cấu hình biến cho mô hình nền (Trend & Calendar)
base_features = [
    'month_sin', 'month_cos', 
    'dow_sin', 'dow_cos', 
    'dom_sin', 'dom_cos', 
    'days_to_tet'
]
# Khởi tạo và huấn luyện Ridge
ridge_rev = Ridge(alpha=1.0)
ridge_cogs = Ridge(alpha=1.0)

ridge_rev.fit(Data[base_features], Data['target_rev'])
ridge_cogs.fit(Data[base_features], Data['target_cogs'])

# Tính toán "Revenue nền" và "Phần dư" (Residuals)
Data['base_rev'] = ridge_rev.predict(Data[base_features])
Data['base_cogs'] = ridge_cogs.predict(Data[base_features])

Data['res_rev'] = Data['target_rev'] - Data['base_rev']
Data['res_cogs'] = Data['target_cogs'] - Data['base_cogs']

# Tính toán mức nền cho tương lai
future_df['base_rev'] = ridge_rev.predict(future_df[base_features])
future_df['base_cogs'] = ridge_cogs.predict(future_df[base_features])

In [ ]:
# Tính trên Log Revenue để ổn định biến động
Data['rev_roll_mean_7d'] = Data['target_rev'].rolling(window=7).mean().shift(1)
Data['rev_roll_std_7d'] = Data['target_rev'].rolling(window=7).std().shift(1)

# Áp dụng tương tự cho future_df (dựa trên các giá trị cuối cùng của tập Train)
last_rev_mean_7d = Data['rev_roll_mean_7d'].iloc[-1]
last_rev_std_7d = Data['rev_roll_std_7d'].iloc[-1]

future_df['rev_roll_mean_7d'] = last_rev_mean_7d
future_df['rev_roll_std_7d'] = last_rev_std_7d

# Tốc độ bán hàng (Revenue / Inventory) - Dùng Lag để tránh rò rỉ dữ liệu
Data['inv_turnover_rate'] = Data['target_rev'].shift(1) / (Data['fill_rate_lag_1m'] + 0.01)

# Dùng giá trị cuối cùng của train cho giai đoạn tương lai
future_df['inv_turnover_rate'] = Data['inv_turnover_rate'].iloc[-1]

# Cường độ Stockout so với Traffic
Data['stockout_per_session'] = Data['stockout_intensity'] / (Data['sessions_lag_365_y'] + 1)

In [ ]:
def double_day_pay_day_traffic_relative_ratio(df):
    # Sao chép để tránh lỗi SettingWithCopyWarning
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    # --- NNGÀY NHẬN LƯƠNG ---
    # Thường là ngày 15 và ngày cuối tháng (30 hoặc 31)
    df['is_payday'] = df['Date'].dt.day.isin([15, 30, 31]).astype(int)
    
    # --- NGÀY ĐÔI ---
    def get_days_to_next_double_day(current_date):
        # Tạo danh sách các ngày đôi trong năm của ngày hiện tại
        double_days = [pd.Timestamp(year=current_date.year, month=i, day=i) for i in range(1, 13)]
        # Lọc các ngày đôi trong tương lai
        future_dd = [d for d in double_days if d >= current_date]
        if not future_dd:
            # Nếu hết ngày đôi trong năm, lấy ngày 1/1 năm sau
            next_dd = pd.Timestamp(year=current_date.year + 1, month=1, day=1)
        else:
            next_dd = future_dd[0]
        return (next_dd - current_date).days

    df['days_to_double_day'] = df['Date'].apply(get_days_to_next_double_day)
    
    # --- TRAFFIC_RELATIVE_RATIO (Tỷ lệ Traffic so với trung bình tháng) ---
    df['traffic_avg_30d'] = df['sessions_lag_365_y'].rolling(window=30, min_periods=1).mean()
    df['traffic_relative_ratio'] = df['sessions_lag_365_y'] / (df['traffic_avg_30d'] + 1)
    
    # Xóa cột trung bình tạm thời để gọn data
    return df.drop(columns=['traffic_avg_30d']).fillna(0)


Data = double_day_pay_day_traffic_relative_ratio(Data)
future_df = double_day_pay_day_traffic_relative_ratio(future_df)

print("Đã xây dựng features: is_payday, days_to_double_day, traffic_relative_ratio")

In [ ]:
xgb_features = [
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'dom_sin', 'dom_cos',
    'days_to_holiday', 'days_to_tet', 'is_tet_season', 
    'tet_proximity_sin', 'tet_proximity_cos',
    'sessions_lag_365_y', 'bounce_rate_lag_365', 'sessions_trend_lag',
    'fill_rate_lag_1m', 'stockout_intensity',
    'promo_probability', 'base_rev', # Thêm base_rev để XGBoost học phần dư
    'rev_roll_mean_7d', 'rev_roll_std_7d', 'inv_turnover_rate',
    'days_to_double_day', 'is_payday', 'traffic_relative_ratio'
]

# Khởi tạo XGBoost
model_res_rev = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,   # Giảm tốc độ học để học kỹ hơn
    max_depth=4,          # Giảm độ sâu của cây 
    reg_alpha=10,         # L1 Regularization 
    reg_lambda=1,         # L2 Regularization
    subsample=0.7,        # Chỉ lấy 70% dữ liệu cho mỗi cây
    colsample_bytree=0.7,  # Chỉ lấy 70% feature cho mỗi cây
    early_stopping_rounds=50, 
)
model_res_cogs = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=4,
    reg_alpha=10,
    reg_lambda=1,
    subsample=0.7,
    colsample_bytree=0.7,
    early_stopping_rounds=50, 
)

# --- CHIẾN THUẬT VALIDATION 2022 ---
# Train: 2012 -> 2021
# Valid: 2022 (Dùng để tinh chỉnh tham số)

train_idx = Data[Data['Date'] < '2022-01-01'].index
valid_idx = Data[(Data['Date'] >= '2022-01-01') & (Data['Date'] <= '2022-12-31')].index

X_train_rev = Data.loc[train_idx, xgb_features]
y_train_rev = Data.loc[train_idx, 'res_rev'].values 

X_valid_rev = Data.loc[valid_idx, xgb_features]
y_valid_rev = Data.loc[valid_idx, 'res_rev'].values 

# Tương tự cho COGS
X_train_cogs = Data.loc[train_idx, xgb_features]
X_valid_cogs = Data.loc[valid_idx, xgb_features]

y_train_cogs = Data.loc[train_idx, 'res_cogs'].values
y_valid_cogs = Data.loc[valid_idx, 'res_cogs'].values


model_res_rev.fit(
    X_train_rev, y_train_rev,
    eval_set=[(X_valid_rev, y_valid_rev )],
    verbose=100
)
model_res_cogs.fit(
    X_train_cogs, y_train_cogs,
    eval_set=[(X_valid_cogs, y_valid_cogs)],
    verbose=100
)

In [ ]:

valid_mask = (Data['Date'] >= '2022-01-01') & (Data['Date'] <= '2022-12-31')
# --- BƯỚC 1: ĐIỀN CÁC FEATURE MỚI CHO TẬP VALIDATE 2022 ---
Data['days_to_double_day'] = Data['Date'].apply(lambda x: 30) # Có thể viết hàm chi tiết sau
Data['traffic_relative_ratio'] = Data['sessions_lag_365_y'] / (Data['sessions_lag_365_y'].rolling(30).mean() + 1)
Data['is_payday'] = Data['Date'].dt.day.isin([15, 30, 31]).astype(int)

# Cập nhật lại X_valid_rev sau khi thêm feature
xgb_features_v2 = xgb_features  
X_valid_rev_v2 = Data.loc[valid_mask, xgb_features_v2]

# --- BƯỚC 2: VÒNG LẶP ---
for depth in [3, 4]: # Thử depth thấp để chống Overfit
    model = XGBRegressor(max_depth=depth, n_estimators=1000, learning_rate=0.01, early_stopping_rounds=50)
    # Train trên tập res_rev
    model.fit(Data.loc[train_idx, xgb_features_v2], y_train_rev, 
              eval_set=[(X_valid_rev_v2, y_valid_rev)], verbose=False)
    
    # Dự báo Hybrid trên tập 2022
    base_pred = ridge_rev.predict(X_valid_rev_v2[base_features])
    res_pred = model.predict(X_valid_rev_v2)
    y_v_pred_log = base_pred + res_pred
    
    # Tính MAE thực tế (VND)
    mae_2022 = mean_absolute_error(np.expm1(y_valid_rev + ridge_rev.predict(X_valid_rev_v2[base_features])), 
                                  np.expm1(y_v_pred_log))
    print(f"Depth {depth} -> MAE 2022: {mae_2022:,.0f} VND")

In [ ]:
final_test = future_df.copy()

history_log_rev = list(Data['target_rev'].tail(7).values)

for i in range(len(final_test)):
    # 1. Tính toán feature ĐỘNG dựa trên kết quả i-1
    final_test.loc[i, 'rev_roll_mean_7d'] = np.mean(history_log_rev[-7:])
    final_test.loc[i, 'rev_roll_std_7d'] = np.std(history_log_rev[-7:])
    final_test.loc[i, 'inv_turnover_rate'] = history_log_rev[-1] / (final_test.loc[i, 'fill_rate_lag_1m'] + 0.01)
    
    # 2. Predict (Dùng danh sách feature đầy đủ v2)
    base_val = ridge_rev.predict(final_test.loc[[i], base_features])[0]
    res_val = model_res_rev.predict(final_test.loc[[i], xgb_features_v2])[0]
    pred_log_rev = base_val + res_val
    
    # 3. Cập nhật kết quả vào chuỗi lịch sử
    final_test.loc[i, 'target_rev_final'] = pred_log_rev
    history_log_rev.append(pred_log_rev)

# Giải mã kết quả
final_test['Revenue'] = np.expm1(final_test['target_rev_final'])
# COGS có thể tính theo tỷ lệ để ổn định score: COGS = Revenue * (mean_cogs_ratio)
mean_ratio = (Data['COGS'] / Data['Revenue']).mean()
final_test['COGS'] = final_test['Revenue'] * mean_ratio

# Xuất file
submission = final_test[['Date', 'Revenue', 'COGS']]
submission.to_csv('Task 3/submissions/submission_ml.csv', index=False)
print("Xuất file thành công!")